**## 0. 공통 유틸 (2026.7.1 광주·전남 통합 반영)**

기존 KOSIS/SGIS 원자료(자동차등록대수 등)는 광주광역시/전라남도를 아직 따로따로 표기하는데, 가연 버스 데이터와 은규 지형장벽 데이터(2026-07-01 경계 기준)는 이미 통합된 "전남광주통합특별시"로 표기돼 있다. `resolve_sido()`를 모든 지역명 처리 지점에 일관되게 넣어두면, 어느 쪽이 옛날 표기든 새 표기든 자동으로 같은 키로 수렴한다(실제로 돌려서 확인함: "광주광역시 동구"/"전라남도 목포시"가 자동으로 "전남광주통합특별시 동구"/"전남광주통합특별시 목포시"로 변환됨).

In [1]:
import pandas as pd
import numpy as np
from google.colab import files

SIDO_NAMES = {
    '서울특별시','부산광역시','대구광역시','인천광역시','광주광역시','대전광역시','울산광역시',
    '세종특별자치시','경기도','강원특별자치도','충청북도','충청남도','전북특별자치도',
    '전라남도','경상북도','경상남도','제주특별자치도','전남광주통합특별시',
}
SIDO_ALIASES = {
    '강원도': '강원특별자치도',
    '전라북도': '전북특별자치도',
    '광주광역시': '전남광주통합특별시',   # 2026.7.1 광주·전남 통합
    '전라남도': '전남광주통합특별시',     # 2026.7.1 광주·전남 통합
}
NON_REGION = {'전국', '동부', '읍부', '면부'}
GU_HAVING_CITIES = {'수원시','성남시','안양시','안산시','고양시','용인시',
                     '청주시','천안시','전주시','포항시','창원시','부천시'}
BARE_GU_TO_CITY = {
    ('경기도','장안구'):'수원시', ('경기도','권선구'):'수원시', ('경기도','팔달구'):'수원시', ('경기도','영통구'):'수원시',
    ('경기도','수정구'):'성남시', ('경기도','중원구'):'성남시', ('경기도','분당구'):'성남시',
    ('경기도','만안구'):'안양시', ('경기도','동안구'):'안양시',
    ('경기도','상록구'):'안산시', ('경기도','단원구'):'안산시',
    ('경기도','덕양구'):'고양시', ('경기도','일산동구'):'고양시', ('경기도','일산서구'):'고양시',
    ('경기도','처인구'):'용인시', ('경기도','기흥구'):'용인시', ('경기도','수지구'):'용인시',
    ('경기도','원미구'):'부천시', ('경기도','소사구'):'부천시', ('경기도','오정구'):'부천시',
    ('충청북도','상당구'):'청주시', ('충청북도','서원구'):'청주시', ('충청북도','흥덕구'):'청주시', ('충청북도','청원구'):'청주시',
    ('충청남도','동남구'):'천안시', ('충청남도','서북구'):'천안시',
    ('전북특별자치도','완산구'):'전주시', ('전북특별자치도','덕진구'):'전주시',
    ('경상북도','남구'):'포항시', ('경상북도','북구'):'포항시',
    ('경상남도','의창구'):'창원시', ('경상남도','성산구'):'창원시',
    ('경상남도','마산합포구'):'창원시', ('경상남도','마산회원구'):'창원시', ('경상남도','진해구'):'창원시',
}
LEGACY_SIDO_OVERRIDE = {
    ('경상북도','군위군'): '대구광역시',   # 2023년 대구 편입
}

def resolve_sido(name):
    """예전 명칭(광주광역시/전라남도/강원도/전라북도)을 현재 명칭으로 통일."""
    if pd.isna(name):
        return None
    return SIDO_ALIASES.get(str(name).strip(), str(name).strip())

def canonical_key(sido, sigungu):
    sido = resolve_sido(sido)
    if sido is None:
        return None
    if '세종' in sido:
        return '세종특별자치시'
    return f"{sido} {sigungu}"

def normalize_sgg_for_join(sido, sgg):
    """특례시 하위구 표기('수원시장안구' 또는 시 이름 없이 '북구'만 있는 경우 등)를 상위 시로 되돌린다."""
    if pd.isna(sgg) or str(sgg).strip() == '':
        return None
    sido_r = resolve_sido(sido)
    name = str(sgg).strip()
    if (sido_r, name) in BARE_GU_TO_CITY:
        return BARE_GU_TO_CITY[(sido_r, name)]
    name_nospace = name.replace(' ', '')
    for city in GU_HAVING_CITIES:
        if name_nospace.startswith(city):
            return city
    return name

def broadcast_sigungu_to_eupmyeon(eupmyeon_df, sigungu_score_df, sido_col='sidonm', sgg_col='sggnm', on='sigungu'):
    """시군구 단위 점수를 읍면동 단위 표에 그대로 복제해서 붙인다 (자가용미보유용)."""
    df = eupmyeon_df.copy()
    def make_key(r):
        sido = resolve_sido(r[sido_col])
        sgg_norm = normalize_sgg_for_join(sido, r.get(sgg_col))
        override = LEGACY_SIDO_OVERRIDE.get((sido, sgg_norm))
        if override:
            sido = override
        return canonical_key(sido, sgg_norm if sgg_norm else sido)
    df['_join_key'] = df.apply(make_key, axis=1)
    merged = df.merge(sigungu_score_df, left_on='_join_key', right_on=on, how='left')
    return merged.drop(columns=['_join_key'])

def build_match_key(sido, sgg, dong):
    """읍면동 고유 데이터끼리(의료·버스 등) 조인할 때 쓰는 공통 키."""
    sido_r = resolve_sido(sido)
    sgg_norm = normalize_sgg_for_join(sido_r, sgg) or sido_r
    return f"{sido_r}|{sgg_norm}|{str(dong).strip()}"

def minmax(s):
    """값이 클수록 나쁨(취약)인 지표를 그대로 0~1로."""
    mn, mx = s.min(), s.max()
    return (s - mn) / (mx - mn)

def minmax_invert(s):
    """값이 클수록 좋음(안 취약)인 지표를 뒤집어서 0~1 취약점수로."""
    mn, mx = s.min(), s.max()
    return 1 - (s - mn) / (mx - mn)

**## 1. 자가용미보유 (시군구 단위 계산)**

In [2]:
print('1) 1인당 자동차등록대수 CSV 업로드')
uploaded = files.upload()
car_path = list(uploaded.keys())[0]

def load_car_percap_sigungu(path):
    df = pd.read_csv(path, encoding='cp949')
    df = df.iloc[1:].reset_index(drop=True)
    df.columns = ['행정구역별', '1인당등록대수']
    df['1인당등록대수'] = pd.to_numeric(df['1인당등록대수'], errors='coerce')

    keys, values = [], []
    cur_sido = None
    skip_sub_gu = False
    for raw, val in zip(df['행정구역별'], df['1인당등록대수']):
        name = str(raw).strip()
        if name == '전남광주통합특별시' and pd.isna(val):
            continue  # 값 없는 표기용 헤더 행 (실제 광주/전라남도는 아래에서 각각 처리됨 -> 자동으로 통합명으로 변환됨)
        if name in NON_REGION:
            continue
        canon = resolve_sido(name)
        if canon in SIDO_NAMES:
            if '세종' in canon and pd.notna(val):
                keys.append('세종특별자치시'); values.append(val)
            cur_sido, skip_sub_gu = canon, False
            continue
        if skip_sub_gu and name.endswith('구'):
            continue
        skip_sub_gu = False
        if name.endswith(('시', '군', '구')):
            if name in GU_HAVING_CITIES:
                skip_sub_gu = True
            if cur_sido:
                keys.append(canonical_key(cur_sido, name)); values.append(val)
            continue

    out = pd.DataFrame({'sigungu': keys, '1인당_자동차등록대수': values})
    return out.groupby('sigungu', as_index=False)['1인당_자동차등록대수'].mean()

car_sigungu = load_car_percap_sigungu(car_path)
car_sigungu['자가용미보유_취약점수'] = minmax_invert(car_sigungu['1인당_자동차등록대수'])
print(car_sigungu.shape, '(광주+전남 통합 후 228개 전후로 나와야 정상)')

1) 1인당 자동차등록대수 CSV 업로드


Saving 1인당_자동차_등록대수_시도_시_군_구__20260923121149.csv to 1인당_자동차_등록대수_시도_시_군_구__20260923121149.csv
(229, 3) (광주+전남 통합 후 228개 전후로 나와야 정상)


**## 2. 읍면동 마스터 테이블 (지형장벽 기준) + 자가용미보유 브로드캐스트**

In [3]:
print('2) slope_score.csv 업로드 (은규 지형장벽, 기준 테이블)')
uploaded = files.upload()
eupmyeon_master = pd.read_csv(list(uploaded.keys())[0])
print('지형장벽 행 수:', len(eupmyeon_master))

eupmyeon_master = broadcast_sigungu_to_eupmyeon(eupmyeon_master, car_sigungu)
print('자가용미보유 결측:', eupmyeon_master['자가용미보유_취약점수'].isna().sum())
if eupmyeon_master['자가용미보유_취약점수'].isna().sum():
    print(eupmyeon_master[eupmyeon_master['자가용미보유_취약점수'].isna()][['sidonm','sggnm','emdnm']].drop_duplicates())

2) slope_score.csv 업로드 (은규 지형장벽, 기준 테이블)


Saving slope_score.csv to slope_score.csv
지형장벽 행 수: 3555
자가용미보유 결측: 77
     sidonm   sggnm emdnm
914   인천광역시     서해구  가정2동
1181  인천광역시     서해구  석남3동
1220    경기도  화성시동탄구  동탄9동
1310    경기도  화성시동탄구  동탄5동
1393  인천광역시     영종구   용유동
...     ...     ...   ...
3471  인천광역시     서해구  청라1동
3473  인천광역시     영종구  운서1동
3482  인천광역시     영종구  영종1동
3498  인천광역시     서해구  청라2동
3542    경기도  화성시만세구   새솔동

[77 rows x 3 columns]


**## 2-보완. 고령인구비율 (KOSIS 101_DT_1IN1503, 읍면동 단위 재다운로드본)**
KOSIS에서 "행정구역별(읍면동)" 트리를 시군구 레벨까지만 펼친 채로 받으면 시군구 단위로만 나온다 — 트리를 읍면동까지 전부 펼치고, 항목을 "총인구(명)" 하나로 좁혀서(그래야 20만 셀 제한을 통과함) 다시 받은 파일이면 진짜 읍면동 단위다. 단, 이 KOSIS 원자료는 시도→시군구→읍면동 각 레벨마다 그 레벨 자체의 합계 행(예: "수원시" 전체 합계가 "장안구" 등 하위 구 목록이 나오기 전에 먼저 한 번 나옴)을 끼워 넣기 때문에, 단순히 "시/군/구로 안 끝나면 읍면동"으로 판단하면 특례시(수원시·성남시·안양시·청주시·천안시·전주시·포항시·창원시·부천시 등)의 시 전체 합계 행을 가짜 읍면동으로 잘못 잡아먹는다. 아래 코드는 연속된 동일 라벨을 하나의 "블록"으로 묶어서 이 문제를 피한다.

In [4]:
print('2-보완) 고령인구 KOSIS 파일 업로드 (101_DT_1IN1503, 읍면동 단위 재다운로드본)')
uploaded = files.upload()
age_path = list(uploaded.keys())[0]

import xml.etree.ElementTree as ET

def read_kosis_xml_xls(path):
    """KOSIS가 .xls 확장자로 내려주지만 실제로는 XML(엑셀 2003 XML) 형식인 파일을 파싱."""
    with open(path, 'rb') as f:
        raw = f.read()
    text = raw.decode('euc-kr', errors='replace').lstrip()
    text = text.replace('encoding="EUC-KR"', 'encoding="UTF-8"')
    idx = text.find('메타정보')
    if idx != -1:
        idx2 = text.rfind('<Worksheet', 0, idx)
        text = text[:idx2] + '</Workbook>'
    root = ET.fromstring(text)
    ns = '{urn:schemas-microsoft-com:office:spreadsheet}'
    rows_out = []
    for ws in root.findall(f'{ns}Worksheet'):
        table = ws.find(f'{ns}Table')
        if table is None:
            continue
        for row in table.findall(f'{ns}Row'):
            cells = []
            for cell in row.findall(f'{ns}Cell'):
                data = cell.find(f'{ns}Data')
                cells.append(data.text if data is not None else None)
            rows_out.append(cells)
    return rows_out

def load_age_eupmyeondong(path):
    raw_rows = read_kosis_xml_xls(path)
    data_rows = [r for r in raw_rows[2:] if r and len(r) >= 5]
    df = pd.DataFrame(data_rows, columns=['행정구역_raw', '연령별', '항목', '단위', '2025'])
    df['2025'] = pd.to_numeric(df['2025'], errors='coerce')

    names = df['행정구역_raw'].astype(str).str.strip()
    block_id = (names != names.shift()).cumsum()        # 연속 동일 라벨을 하나의 블록으로
    block_labels = names.groupby(block_id).first()

    cur_sido, cur_sgg, cur_sgg_is_gu_city = None, None, False
    block_sido, block_sgg, block_dong = {}, {}, {}

    for bid, name in block_labels.items():
        if name in NON_REGION:
            block_sido[bid] = block_sgg[bid] = block_dong[bid] = None
            continue
        canon = resolve_sido(name)
        if canon in SIDO_NAMES:
            cur_sido, cur_sgg, cur_sgg_is_gu_city = canon, None, False
            block_sido[bid] = block_sgg[bid] = block_dong[bid] = None
            continue
        if name.endswith('구') and cur_sgg_is_gu_city:
            # 특례시 하위 자치구(예: 수원시 장안구) -> 구조 표시일 뿐, 실제 단위 아님
            block_sido[bid] = block_sgg[bid] = block_dong[bid] = None
            continue
        if name.endswith(('시', '군', '구')):
            cur_sgg = name
            cur_sgg_is_gu_city = name in GU_HAVING_CITIES
            block_sido[bid] = block_sgg[bid] = block_dong[bid] = None
            continue
        # 읍면동 레벨
        block_sido[bid] = cur_sido
        block_sgg[bid] = cur_sgg
        block_dong[bid] = name

    df['_block'] = block_id
    df['sido'] = df['_block'].map(block_sido)
    df['sgg'] = df['_block'].map(block_sgg)
    df['dong'] = df['_block'].map(block_dong)

    valid = df[df['dong'].notna()].copy()
    total = valid[valid['연령별'] == '합계'].groupby(['sido', 'sgg', 'dong'])['2025'].sum().reset_index(name='전체인구')
    elderly = valid[valid['연령별'] == '65세이상'].groupby(['sido', 'sgg', 'dong'])['2025'].sum().reset_index(name='고령인구')
    out = total.merge(elderly, on=['sido', 'sgg', 'dong'], how='left')
    out['고령인구비율'] = out['고령인구'] / out['전체인구']
    return out

age_eupmyeon = load_age_eupmyeondong(age_path)
print(age_eupmyeon.shape, '(3,555~3,558개 근처여야 정상)')
print('전체인구 합계 (전국 51,817,499명과 비교):', age_eupmyeon['전체인구'].sum())

age_eupmyeon['_match_key'] = age_eupmyeon.apply(lambda r: build_match_key(r['sido'], r['sgg'], r['dong']), axis=1)
eupmyeon_master['_match_key'] = eupmyeon_master.apply(lambda r: build_match_key(r['sidonm'], r['sggnm'], r['emdnm']), axis=1)

eupmyeon_master = eupmyeon_master.merge(
    age_eupmyeon[['_match_key', '고령인구비율']], on='_match_key', how='left'
)
eupmyeon_master = eupmyeon_master.drop(columns=['_match_key'])
eupmyeon_master['고령인구비율_취약점수'] = minmax(eupmyeon_master['고령인구비율'])  # 비율이 높을수록 취약

print('고령인구비율 결측:', eupmyeon_master['고령인구비율'].isna().sum())
if eupmyeon_master['고령인구비율'].isna().sum():
    print(eupmyeon_master[eupmyeon_master['고령인구비율'].isna()][['sidonm', 'sggnm', 'emdnm']].drop_duplicates().head(20))

2-보완) 고령인구 KOSIS 파일 업로드 (101_DT_1IN1503, 읍면동 단위 재다운로드본)


Saving 101_DT_1IN1503_20261004030052.xls to 101_DT_1IN1503_20261004030052.xls
(3557, 6) (3,555~3,558개 근처여야 정상)
전체인구 합계 (전국 51,817,499명과 비교): 51817499.0
고령인구비율 결측: 85
     sidonm   sggnm  emdnm
128     경기도  안양시만안구   병목안동
847   부산광역시    부산진구   당감1동
886     경기도  안양시만안구    명학동
914   인천광역시     서해구   가정2동
1088    경기도  용인시처인구    양지읍
1181  인천광역시     서해구   석남3동
1220    경기도  화성시동탄구   동탄9동
1310    경기도  화성시동탄구   동탄5동
1393  인천광역시     영종구    용유동
1510  인천광역시     서해구   가정3동
1586    경기도  화성시동탄구   동탄8동
1636    경기도  화성시효행구    봉담읍
1655  인천광역시     검단구  불로대곡동
1670  인천광역시     검단구   아라2동
1671  인천광역시     서해구    연희동
1682    경기도  화성시효행구    비봉면
1713  인천광역시     검단구    검단동
1748  인천광역시     서해구   가좌2동
1749    경기도  화성시동탄구   동탄7동
1891    경기도  화성시효행구    매송면


**## 3. 의료접근성 (HIRA 버전, 동 포함 전국 3,558개 — 0 대체 없음)**

팀에서 이미 만든 `medical_access_score_hira.csv`(저장소 경로 `data/processed/medical/`)는 동 지역까지 전국 읍면동 전부에 실제 계산값이 들어 있다. 예전에 쓰던 `medical_access_score.csv`는 마을조사표 기반이라 읍/면 1,404개뿐이어서 동 지역을 0(가장 양호)으로 채워야 했는데, 이 0 대체가 의료_취약점수의 엔트로피 가중치를 58.9%까지 끌어올린 원인으로 의심됐다. HIRA 버전을 쓰면 0 대체가 필요 없다.

HIRA 파일의 실제 컬럼은 `읍면_key, sidonm, sggnm, emdnm, 1차의료_최근접거리_km, 2차의료_최근접거리_km, 1차의료_취약점수_HIRA, 2차의료_취약점수_HIRA, 의료시설_취약점수_HIRA`이다. 최종 의료 점수는 이미 계산돼 있는 `의료시설_취약점수_HIRA`이고, 확인해보니 0.4 × 1차 + 0.6 × 2차와 완전히 일치한다(최대 차이 1e-16). 따라서 가중치를 코드에서 다시 정할 필요 없이 이 컬럼을 그대로 쓰면 된다. 이 파일은 `sidonm`/`sggnm`/`emdnm`이 은규 지형장벽 파일과 같은 2026.7.1 기준 표기(전남광주통합특별시, 서해구, 동탄구 등)라서 `읍면_key`로 바로 붙는다.

In [5]:
print('3) medical_access_score_hira.csv 업로드 (HIRA 기반 의료접근성, 동 포함 전국 3,558개)')
uploaded = files.upload()
medical_df = pd.read_csv(list(uploaded.keys())[0], encoding='utf-8-sig')
print('행 수:', len(medical_df), '/ 읍면_key 중복:', medical_df['읍면_key'].duplicated().sum())

# 0.4/0.6이 실제로 적용돼 있는지 확인 (0에 가까워야 정상)
check = (medical_df['의료시설_취약점수_HIRA']
         - (0.4 * medical_df['1차의료_취약점수_HIRA'] + 0.6 * medical_df['2차의료_취약점수_HIRA'])).abs().max()
print('0.4/0.6 재계산과의 최대 차이:', check)

medical_df = medical_df.rename(columns={'의료시설_취약점수_HIRA': '의료_취약점수'})
eupmyeon_master = eupmyeon_master.merge(
    medical_df[['읍면_key', '의료_취약점수']], on='읍면_key', how='left'
)
# 더 이상 fillna(0)을 하지 않는다. 결측이 남으면 그대로 확인한다.
print('의료_취약점수 결측:', eupmyeon_master['의료_취약점수'].isna().sum())
if eupmyeon_master['의료_취약점수'].isna().sum():
    print(eupmyeon_master[eupmyeon_master['의료_취약점수'].isna()][['sidonm', 'sggnm', 'emdnm']].drop_duplicates().head(20))

3) medical_access_score_hira.csv 업로드 (HIRA 기반 의료접근성, 동 포함 전국 3,558개)


Saving medical_access_score_hira.csv to medical_access_score_hira.csv
행 수: 3558 / 읍면_key 중복: 0
0.4/0.6 재계산과의 최대 차이: 1.1102230246251565e-16
의료_취약점수 결측: 0


**## 4. 버스 (정류장 인구대비 + 배차간격, 가연 2차 수정본)**
가연이 시도명 정규화(광주·전남 통합 반영)를 적용해서 올려준 최신 파일. `province`/`sigungu`/`dong`/`dong_code`/`population_status` 컬럼이 있고 `per10k`(정류장, 인구대비), `headway_min`(배차간격)을 바로 쓸 수 있다.

In [7]:
print('4-1) stop_supply_by_dong.csv 업로드 (가연 버스정류장·인구대비, 2차 수정본)')
uploaded = files.upload()
stop_df = pd.read_csv(list(uploaded.keys())[0])

print('4-2) dong_supply.csv 업로드 (가연 배차간격, 2차 수정본)')
uploaded = files.upload()
headway_df = pd.read_csv(list(uploaded.keys())[0])

stop_df['버스정류장_취약점수'] = minmax_invert(stop_df['per10k'])          # 적을수록 취약
headway_df['배차간격_취약점수'] = minmax(headway_df['headway_min'])        # 길수록 취약

stop_df['_match_key'] = stop_df.apply(lambda r: build_match_key(r['province'], r['sigungu'], r['dong']), axis=1)
headway_df['_match_key'] = headway_df.apply(lambda r: build_match_key(r['province'], r['sigungu'], r['dong']), axis=1)
eupmyeon_master['_match_key'] = eupmyeon_master.apply(lambda r: build_match_key(r['sidonm'], r['sggnm'], r['emdnm']), axis=1)

eupmyeon_master = eupmyeon_master.merge(
    stop_df[['_match_key', '버스정류장_취약점수']], on='_match_key', how='left'
).merge(
    headway_df[['_match_key', '배차간격_취약점수']], on='_match_key', how='left'
)
eupmyeon_master = eupmyeon_master.drop(columns=['_match_key'])

print('버스정류장_취약점수 결측 (대체 전):', eupmyeon_master['버스정류장_취약점수'].isna().sum())
print('배차간격_취약점수 결측 (대체 전):', eupmyeon_master['배차간격_취약점수'].isna().sum())

# 결측 여부를 먼저 플래그로 남겨둠 (보고서용)
eupmyeon_master['버스정류장_취약점수_대체여부'] = eupmyeon_master['버스정류장_취약점수'].isna()
eupmyeon_master['배차간격_취약점수_대체여부'] = eupmyeon_master['배차간격_취약점수'].isna()

def fill_by_region_mean(df, col, sgg_col='sggnm', sido_col='sidonm'):
    sgg_mean = df.groupby(sgg_col)[col].transform('mean')
    df[col] = df[col].fillna(sgg_mean)
    sido_mean = df.groupby(sido_col)[col].transform('mean')
    df[col] = df[col].fillna(sido_mean)
    return df

eupmyeon_master = fill_by_region_mean(eupmyeon_master, '버스정류장_취약점수')
eupmyeon_master = fill_by_region_mean(eupmyeon_master, '배차간격_취약점수')

print('버스정류장_취약점수 결측 (대체 후, 0이어야 정상):', eupmyeon_master['버스정류장_취약점수'].isna().sum())
print('배차간격_취약점수 결측 (대체 후, 0이어야 정상):', eupmyeon_master['배차간격_취약점수'].isna().sum())
print('잠정 대체된 읍면동 수 — 버스정류장:', eupmyeon_master['버스정류장_취약점수_대체여부'].sum(),
      ' / 배차간격:', eupmyeon_master['배차간격_취약점수_대체여부'].sum())

4-1) stop_supply_by_dong.csv 업로드 (가연 버스정류장·인구대비, 2차 수정본)


Saving stop_supply_by_dong.csv to stop_supply_by_dong.csv
4-2) dong_supply.csv 업로드 (가연 배차간격, 2차 수정본)


Saving dong_supply.csv to dong_supply.csv
버스정류장_취약점수 결측 (대체 전): 144
배차간격_취약점수 결측 (대체 전): 13
버스정류장_취약점수 결측 (대체 후, 0이어야 정상): 0
배차간격_취약점수 결측 (대체 후, 0이어야 정상): 0
잠정 대체된 읍면동 수 — 버스정류장: 144  / 배차간격: 13


**## 5. 엔트로피 가중치 계산**

In [8]:
# 공급공백(완성되면) 나오는 대로 여기 추가
final_vars = ['지형장벽_취약점수', '자가용미보유_취약점수', '의료_취약점수',
              '버스정류장_취약점수', '배차간격_취약점수', '고령인구비율_취약점수']

# 엔트로피 계산 전 필수 점검: 여기서 하나라도 0이 아니면, entropy_weights의
# dropna()가 그 행들을 통째로 날려서 가중치 자체가 왜곡될 수 있음 -> 반드시 확인
print('엔트로피 계산 직전 변수별 결측 개수:')
print(eupmyeon_master[final_vars].isna().sum())

def entropy_weights(df, cols, eps=1e-6):
    X = df[cols].copy().dropna()
    X = X * (1 - 2*eps) + eps
    P = X / X.sum(axis=0)
    n = len(X)
    E = -(P * np.log(P)).sum(axis=0) / np.log(n)
    D = 1 - E
    W = D / D.sum()
    return W

weights = entropy_weights(eupmyeon_master, final_vars)
print(weights)

eupmyeon_master['이동격차지수'] = sum(eupmyeon_master[v].fillna(0) * weights[v] for v in final_vars)

final_ranked = eupmyeon_master[~eupmyeon_master['sidonm'].str.contains('서울')].copy()
final_ranked = final_ranked.sort_values('이동격차지수', ascending=False)
print(final_ranked[['읍면_key'] + final_vars + ['이동격차지수']].head(20))

엔트로피 계산 직전 변수별 결측 개수:
지형장벽_취약점수       0
자가용미보유_취약점수    77
의료_취약점수         0
버스정류장_취약점수      0
배차간격_취약점수       0
고령인구비율_취약점수    85
dtype: int64
지형장벽_취약점수      0.160038
자가용미보유_취약점수    0.005189
의료_취약점수        0.336894
버스정류장_취약점수     0.010236
배차간격_취약점수      0.399321
고령인구비율_취약점수    0.088322
dtype: float64
               읍면_key  지형장벽_취약점수  자가용미보유_취약점수   의료_취약점수  버스정류장_취약점수  \
755      경상남도 하동군 북천면   0.512814     0.878788  0.143456    0.740569   
79       경상북도 영양군 수비면   0.770311     0.848485  0.292192    0.528027   
1104     경상남도 밀양시 초동면   0.430421     0.878788  0.148001    0.919073   
122      경상북도 성주군 가천면   0.734061     0.818182  0.273239    0.838599   
119      경상북도 영양군 청기면   0.734749     0.848485  0.302812    0.463525   
103   강원특별자치도 강릉시 왕산면   0.745806     0.878788  0.179558    0.527902   
62       충청북도 영동군 용화면   0.793058     0.878788  0.251777    0.475824   
760      경상북도 봉화군 상운면   0.511653     0.848485  0.119088    0.705873   
192   강원특별자치도 평창군 봉평면   0.690805     0.848485  0.310582    

**## 6. 동일가중치와 비교 (민감도분석)**

In [9]:
equal_w = {v: 1/len(final_vars) for v in final_vars}
eupmyeon_master['이동격차지수_동일가중'] = sum(
    eupmyeon_master[v].fillna(0) * equal_w[v] for v in final_vars
)

comparison = eupmyeon_master[['읍면_key']].copy()
comparison['순위_엔트로피'] = eupmyeon_master['이동격차지수'].rank(ascending=False)
comparison['순위_동일가중'] = eupmyeon_master['이동격차지수_동일가중'].rank(ascending=False)
comparison['순위차이'] = (comparison['순위_엔트로피'] - comparison['순위_동일가중']).abs()

print('상위 20위 안에서 순위가 가장 많이 바뀐 지역:')
print(comparison.sort_values('순위차이', ascending=False).head(20))

상위 20위 안에서 순위가 가장 많이 바뀐 지역:
                 읍면_key  순위_엔트로피  순위_동일가중    순위차이
2305  전남광주통합특별시 신안군 장산면    649.0   3403.0  2754.0
1854  전남광주통합특별시 신안군 임자면    816.0   3395.0  2579.0
2224  전남광주통합특별시 신안군 비금면    919.0   3443.0  2524.0
2084  전남광주통합특별시 신안군 지도읍    929.0   3446.0  2517.0
2784  전남광주통합특별시 신안군 증도면    988.0   3460.0  2472.0
1382  전남광주통합특별시 신안군 암태면   1044.0   3458.0  2414.0
1379  전남광주통합특별시 신안군 도초면    932.0   3337.0  2405.0
2547  전남광주통합특별시 신안군 안좌면   1098.0   3475.0  2377.0
1690  전남광주통합특별시 신안군 신의면   1136.0   3473.0  2337.0
1958  전남광주통합특별시 신안군 자은면   1179.0   3477.0  2298.0
1871  전남광주통합특별시 신안군 하의면    743.0   2989.0  2246.0
2633  전남광주통합특별시 신안군 팔금면   1366.0   3476.0  2110.0
3407  전남광주통합특별시 해남군 산이면   1331.0   3406.0  2075.0
1825      인천광역시 옹진군 영흥면   1234.0   3245.0  2011.0
2174  전남광주통합특별시 신안군 압해읍   1476.0   3478.0  2002.0
2435     경기도 화성시만세구 송산면   1560.0   3481.0  1921.0
1777  전남광주통합특별시 해남군 해남읍   1346.0   3251.0  1905.0
2497  전남광주통합특별시 해남군 황산면   1091.0   2969.0  1878.0
2940     경기도 화성시만세구 우정

**## 7. 아직 남은 것**

- 공급공백: 마을조사표 기반 지표, 아직 미완성
- 의료를 HIRA 버전(0.4/0.6)으로 바꿨으므로 5·6번을 다시 돌려야 한다. 동 지역 0 대체가 사라지면 의료_취약점수의 엔트로피 가중치(기존 58.9%)와 엔트로피 vs 동일가중치 순위 차이(신안군·해남군 쏠림)가 크게 달라질 수 있어서, 이전 1차·2차 결과 정리본의 의료 관련 해석은 새 결과로 대체해야 한다.
- 은규 데이터(지형장벽/의료)의 `sidonm`/`sggnm` 값이 2026.7.1 통합 이전 표기(광주광역시/전라남도)인지 이후 표기(전남광주통합특별시)인지 확인 — `resolve_sido()`가 양쪽 다 같은 키로 수렴시켜주니 코드 자체는 그대로 둬도 되지만, 결측 체크에서 이상 패턴이 보이면 이 부분부터 의심할 것
- 버스(정류장·배차간격) 146개 읍면동은 가연의 인구 원본에 시군구 정보가 없어 생기는 구조적 한계(동명 중복 136개, 미대응 10개)로, 4번에서 시군구→시도 평균으로 잠정 대체함. 가연이 시군구 포함된 더 나은 인구 데이터를 구하면 그걸로 다시 매칭해서 대체 처리를 없애는 게 이상적 — 최종 보고서에는 "N개 읍면동은 지역 평균으로 대체"라고 명시할 것
- 위 항목들 완성되면 `final_vars` 리스트에 추가하고 그대로 다시 돌리면 됨

## 8. 경기도 이동격차 순위와 똑버스 운행현황 대조

앞의 최종 지수 계산을 실행한 다음 실행합니다. 기존 Colab에 계산 결과가 남아 있으면 아래 코드 셀만 복사해도 됩니다.
전국 기준 가중치와 점수를 그대로 유지하고 경기도만 추출합니다. 경기도만으로 가중치를 다시 계산하지 않습니다.

제공된 운행현황 CSV 사본을 코드에 포함했습니다. 현재 시점의 운행을 별도로 확인한 자료가 아닙니다.
시군과 읍면동 명칭이 운행구역에 명시된 경우만 확인으로 표시합니다. 신도시·지구명, 법정동/행정동 차이와 개편 명칭은 자동 추정하지 않습니다.
‘미확인’ 또는 ‘기록 없음’은 미운행 판정이 아닙니다. 명칭이 일치해도 읍면동 전역을 운행한다는 뜻은 아닙니다.
이 자료에는 승객·탑승 기록이 없어 주민의 실제 이용 여부나 이용률을 판단할 수 없습니다.

결측 포함 지역은 잠정 순위에 주의 표시를 붙입니다. 결측 없는 지역만의 순위도 함께 제공하지만 기존 가중치는 그대로이므로 데이터 문제를 모두 해결한 결과는 아닙니다.
상위 10/20/50은 점수 순 정렬 후 해당 행 수를 선택합니다(경계 동점의 전원 포함 방식이 아님).


In [ ]:
"""최종 노트북의 eupmyeon_master를 받아 경기도와 똑버스 CSV를 비교한다."""
import re
from pathlib import Path
import pandas as pd


def compare_gyeonggi_drt(master, drt, output_dir='gyeonggi_drt_comparison'):
    required = ['sidonm', 'sggnm', 'emdnm', '이동격차지수']
    missing = set(required) - set(master.columns)
    if missing:
        raise ValueError(f'최종 지수 계산을 먼저 실행하세요. 누락 열: {missing}')
    gg = master.loc[master['sidonm'].eq('경기도')].copy()
    if gg.empty:
        raise ValueError('경기도 행이 없습니다.')
    # 전국에서 계산된 점수와 가중치를 유지한 채 경기도만 추출한다.
    gg['비교_시군'] = gg['sggnm'].str.extract(r'^(.+?[시군])', expand=False)
    gg['경기도순위_잠정'] = gg['이동격차지수'].rank(method='min', ascending=False).astype('Int64')
    score_cols = [c for c in master.columns if c.endswith('_취약점수')]
    gg['결측항목'] = gg[score_cols].apply(lambda row: ', '.join(row.index[row.isna()]), axis=1)
    gg['순위주의'] = gg['결측항목'].ne('')
    gg['결측없는지역내순위'] = gg.loc[~gg['순위주의'], '이동격차지수'].rank(method='min', ascending=False).astype('Int64')
    drt = drt.loc[drt['지자체'].ne('계')].copy()
    drt['비교_시군'] = drt['지자체'].str.extract(r'^(.+?[시군])', expand=False)
    # 원문에 읍/면 접미사가 생략된 두 표현만 명시적으로 확장한다.
    expansions = {'은현·남면·덕정역': '은현면·남면·덕정역', '남사·이동읍': '남사읍·이동읍',
                  '광교1·2동': '광교1동·광교2동'}
    def tokens(area):
        area = expansions.get(str(area), str(area))
        return set(re.findall(r'[가-힣0-9]+(?:읍|면|동)(?=$|[\s,·()])', area))
    drt['명시지역'] = drt['운행구역'].map(tokens)
    def match(row):
        city_rows = drt.loc[drt['비교_시군'].eq(row['비교_시군'])]
        hits = city_rows.loc[city_rows['명시지역'].map(lambda x: row['emdnm'] in x)]
        if len(hits):
            status = '운행구역에 읍면동 명시'
        elif len(city_rows):
            status = '읍면동 미확인(해당 시군 운행 기록 있음)'
        else:
            status = '제공 CSV에 해당 시군 기록 없음'
        return pd.Series({
            'DRT대조결과': status,
            '일치_운행구역': ' / '.join(hits['운행구역'].astype(str)),
            '일치_운행개시일': ' / '.join(hits['운행개시일'].astype(str)),
            '시군전체_운행구역': ' / '.join(city_rows['운행구역'].astype(str)),
        })
    gg = pd.concat([gg, gg.apply(match, axis=1)], axis=1)
    gg = gg.sort_values('이동격차지수', ascending=False)
    front = ['경기도순위_잠정', 'sggnm', 'emdnm', '이동격차지수', 'DRT대조결과',
             '일치_운행구역', '일치_운행개시일', '순위주의', '결측항목', '결측없는지역내순위']
    gg = gg[front + [c for c in gg.columns if c not in front]]
    summaries = []
    for label, subset in [('전체', gg), ('결측없는지역', gg.loc[~gg['순위주의']])]:
        for k in (10, 20, 50):
            top = subset.head(k)
            confirmed = top['DRT대조결과'].eq('운행구역에 읍면동 명시').sum()
            summaries.append({'대상': label, '상위N': k, '실제지역수': len(top),
                              '읍면동명시운행지역수': int(confirmed),
                              '명시확인비율': confirmed / len(top) if len(top) else float('nan'),
                              '결측포함지역수': int(top['순위주의'].sum())})
    summary = pd.DataFrame(summaries)
    output = Path(output_dir)
    output.mkdir(parents=True, exist_ok=True)
    gg.to_csv(output / '경기도_이동격차순위_DRT대조.csv', index=False, encoding='utf-8-sig')
    gg.head(20).to_csv(output / '경기도_상위20_DRT대조.csv', index=False, encoding='utf-8-sig')
    summary.to_csv(output / '경기도_상위권_DRT요약.csv', index=False, encoding='utf-8-sig')
    return gg, summary


from io import StringIO
drt_source = pd.read_csv(StringIO('지자체,운행구역,운행대수,운행개시일,추진근거\n계,19개 市,267대,,\n파주시 (24대),운정신도시·교하지구,15대,2021년 12월,규제샌드박스\n파주시 (24대),광탄면,3대,2023년 12월,농어촌\n파주시 (24대),탄현면,3대,2023년 12월,농어촌\n파주시 (24대),운정4동,3대,2023년 12월,대중교통취약지구\n안산시 (16대),대부도,6대,2023년 3월,규제샌드박스\n안산시 (16대),스마트허브,10대,2024년 7월,규제샌드박스\n평택시,고덕국제신도시,15대,2023년 4월,규제샌드박스\n수원시 (30대),광교1·2동,10대,2023년 5월,규제샌드박스\n수원시 (30대),고색동(델타플렉스),10대,2024년 10월,규제샌드박스\n수원시 (30대),당수동,10대,2025년 6월,규제샌드박스\n고양시 (14대),고봉동,3대,2023년 6월,규제샌드박스\n고양시 (14대),식사동,4대,2023년 6월,규제샌드박스\n고양시 (14대),향동동,4대,2024년 9월,규제샌드박스\n고양시 (14대),덕은동,3대,2024년 11월,규제샌드박스\n화성시 (35대),동탄신도시,20대,2023년 6~7월,규제샌드박스\n화성시 (35대),향남1·2지구,5대,2023년 6월,규제샌드박스\n화성시 (35대),봉담읍,5대,2024년 9월,농어촌\n화성시 (35대),남양읍·마도면,5대,2024년 9월,농어촌\n김포시,"고촌읍, 풍무동",10대,2023년 6월,규제샌드박스\n양주시 (25대),"옥정신도시, 삼숭동, 회천지구",15대,2023년 7월,규제샌드박스\n양주시 (25대),백석읍·광적면·양주역,4대,2025년 6월,농어촌\n양주시 (25대),은현·남면·덕정역,3대,2025년 6월,농어촌\n양주시 (25대),장흥면,3대,2025년 6월,농어촌\n하남시 (9대),감일동,5대,2023년 8월,규제샌드박스\n하남시 (9대),위례동,4대,2023년 8월,규제샌드박스\n안성시 (15대),"서부권(공도읍,양성면,원곡면)",4대,2023년 10월,농어촌\n안성시 (15대),"동부권(일죽면,죽산면,삼죽면)",4대,2023년 10월,농어촌\n안성시 (15대),"북부권(보개면,대덕면,고삼면,양성면)",4대,2024년 5월,농어촌\n안성시 (15대),"남부권(금광면,서운면,미양면,보개면)",3대,2024년 5월,농어촌\n이천시 (22대),"시내권(관고동,중리동,증포동,창전동)",12대,2023년 12월,규제샌드박스\n이천시 (22대),장호원읍,7대,2023년 12월,농어촌\n이천시 (22대),율면,3대,2023년 12월,농어촌\n부천시 (7대),"고강본동, 고강1동",3대,2024년 4월,규제샌드박스\n부천시 (7대),"옥길동, 범박동",4대,2024년 4월,규제샌드박스\n광주시 (15대),곤지암읍,4대,2024년 5월,농어촌\n광주시 (15대),도척면,4대,2024년 5월,농어촌\n광주시 (15대),초월읍,4대,2024년 6월,농어촌\n광주시 (15대),퇴촌면,3대,2024년 6월,농어촌\n여주시 (10대),세종대왕면·중앙동·오학동,8대,2024년 6월,농어촌\n여주시 (10대),북내면·강천면,2대,2024년 6월,농어촌\n안양시,안양9동,2대,2024년 8월,규제샌드박스\n연천군,"연천읍, 내산리",2대,2024년 8월,농어촌\n의왕시,초평동,2대,2025년 6월,규제샌드박스\n용인시,남사·이동읍,6대,2025년 6월,농어촌\n의정부시,민락1지구,4대,2025년 6월,규제샌드박스\n의정부시,민락2·고산지구,4대,2025년 6월,규제샌드박스\n'))

gyeonggi_drt, gyeonggi_drt_summary = compare_gyeonggi_drt(eupmyeon_master, drt_source)
display(gyeonggi_drt.head(20)[['경기도순위_잠정', 'sggnm', 'emdnm', '이동격차지수', 'DRT대조결과', '일치_운행구역', '순위주의']])
display(gyeonggi_drt_summary)


In [ ]:
# 비교 결과 CSV 3개를 ZIP으로 내려받기 (Colab)
import shutil
shutil.make_archive('gyeonggi_drt_comparison', 'zip', 'gyeonggi_drt_comparison')
from google.colab import files
files.download('gyeonggi_drt_comparison.zip')
